# Linear algebra, fitting and PCA · Modules 2–4
Run top to bottom with Python 3 and NumPy. Predict the numbers before executing each cell. The demonstrations are complete; the experiment prompts ask you to change one thing and explain the result.

Keep the same train/validation/test split while comparing representations. Synthetic bars teach mechanics, not real-image performance.


In [1]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(205)
x = np.array([3.,4.]); u = np.array([1.,1.])
projection = u * (u @ x) / (u @ u)
residual = x-projection
assert np.allclose(u @ residual,0)
print('length:',np.linalg.norm(x),'projection:',projection,'residual:',residual)
A = np.array([[2.,1.],[0.,1.]])
print('Ax:', A @ np.array([1.,2.]))
B = np.array([[1.,1.],[2.,2.]])
assert np.allclose(B @ np.array([1.,-1.]),0)
print('B rank:',np.linalg.matrix_rank(B))


length: 5.0 projection: [3.5 3.5] residual: [-0.5  0.5]
Ax: [4. 2.]
B rank: 1


## Predict and explain
Change x so it is perpendicular to u. Predict its projection. Then make a full-rank B and explain which nonzero null vector disappears.

## Least squares
Write the design matrix and solve the worked line fit. Residual orthogonality is a check on this known mathematical case.


In [2]:
t = np.array([0.,1.,2.]); b = np.array([1.,2.,2.])
design = np.column_stack([t,np.ones_like(t)])
theta,_,rank,singular = np.linalg.lstsq(design,b,rcond=None)
residual = design @ theta-b
assert np.allclose(theta,[.5,7/6])
assert np.allclose(design.T @ residual,0,atol=1e-12)
print('slope, intercept:',theta,'MSE:',np.mean(residual**2))
print('rank and singular values:',rank,singular)
for outlier in [0.,3.]:
    observations=b.copy();observations[-1]+=outlier
    for penalty in [0.,1.,4.]:
        # The intercept is deliberately unpenalized.
        augmented=np.vstack([design,[np.sqrt(penalty),0.]])
        targets=np.r_[observations,0.]
        fit=np.linalg.lstsq(augmented,targets,rcond=None)[0]
        print('outlier, lambda, fit:',outlier,penalty,fit)
print('condition diag(1,.001):',np.linalg.cond(np.diag([1.,.001])))


slope, intercept: [0.5    1.1667] MSE: 0.055555555555555525
rank and singular values: 2 [2.6762 0.9153]
outlier, lambda, fit: 0.0 0.0 [0.5    1.1667]
outlier, lambda, fit: 0.0 1.0 [0.3333 1.3333]
outlier, lambda, fit: 0.0 4.0 [0.1667 1.5   ]
outlier, lambda, fit: 3.0 0.0 [2.     0.6667]
outlier, lambda, fit: 3.0 1.0 [1.3333 1.3333]
outlier, lambda, fit: 3.0 4.0 [0.6667 2.    ]
condition diag(1,.001): 1000.0


## Image SVD
This factors one pixel matrix. PCA below instead factors centered samples. Check the storage calculation and the sum of discarded squared singular values for each k.


In [3]:
image=np.zeros((12,12));image[2:10,3:5]=1;image[6:8,1:11]+=.6
image+=np.linspace(0,.2,12)[None,:]
U,s,Vt=np.linalg.svd(image,full_matrices=False)
for k in [1,2,3,6]:
    reconstruction=(U[:,:k]*s[:k]) @ Vt[:k]
    error=np.sum((image-reconstruction)**2)
    assert np.allclose(error,np.sum(s[k:]**2),atol=1e-12)
    print('k, squared error, factor storage:',k,error,k*(12+12+1))
print('original storage:',image.size)


k, squared error, factor storage: 1 2.4811459163321494 25
k, squared error, factor storage: 2 0.2477495630789079 50
k, squared error, factor storage: 3 2.582546869976898e-29 75
k, squared error, factor storage: 6 2.586490211538031e-29 150
original storage: 144


## A training-only recognition pipeline
Generate independent noisy 8x8 bars, split once, and fit every representation using training data. Each row is one flattened image. This is a deliberately small nearest-centroid baseline.


In [4]:
def bars(n):
    y=rng.integers(0,2,n);images=np.zeros((n,8,8))
    for i,label in enumerate(y):
        position=int(rng.integers(1,6))
        if label==0: images[i,position:position+2,:]=1.
        else: images[i,:,position:position+2]=1.
    images+=rng.normal(0,.12,images.shape)
    return images.reshape(n,-1),y
X,y=bars(240)
Xtr,Xval,Xtest=X[:160],X[160:200],X[200:]
ytr,yval,ytest=y[:160],y[160:200],y[200:]
mu=Xtr.mean(axis=0)
_,singular,Vt=np.linalg.svd(Xtr-mu,full_matrices=False)
def centroid_predict(train,labels,new):
    centers=np.stack([train[labels==c].mean(axis=0) for c in [0,1]])
    distances=np.sum((new[:,None,:]-centers[None,:,:])**2,axis=2)
    return distances.argmin(axis=1)
raw_val=np.mean(centroid_predict(Xtr,ytr,Xval)==yval)
centers=np.stack([Xtr[ytr==c].mean(axis=0) for c in [0,1]])
assert np.all(np.linalg.norm(centers,axis=1)>0) and np.all(np.linalg.norm(Xval,axis=1)>0)
unit_centers=centers/np.linalg.norm(centers,axis=1,keepdims=True)
unit_validation=Xval/np.linalg.norm(Xval,axis=1,keepdims=True)
cosine_prediction=(unit_validation @ unit_centers.T).argmax(axis=1)
print('Euclidean/cosine validation accuracy:',raw_val,np.mean(cosine_prediction==yval))
choices=[2,4,8,16];scores=[]
for k in choices:
    basis=Vt[:k].T
    train=(Xtr-mu) @ basis;validation=(Xval-mu) @ basis
    score=np.mean(centroid_predict(train,ytr,validation)==yval)
    scores.append(score)
    print('components, validation accuracy:',k,score)
selected=choices[int(np.argmax(scores))]
basis=Vt[:selected].T
test_prediction=centroid_predict((Xtr-mu) @ basis,ytr,(Xtest-mu) @ basis)
print('raw validation:',raw_val,'selected k:',selected)
print('fixed PCA procedure test accuracy:',np.mean(test_prediction==ytest),'test count:',len(ytest))
assert (Xtest-mu).shape[1]==basis.shape[0]


Euclidean/cosine validation accuracy: 1.0 1.0
components, validation accuracy: 2 0.55
components, validation accuracy: 4 0.575
components, validation accuracy: 8 1.0
components, validation accuracy: 16 1.0
raw validation: 1.0 selected k: 8
fixed PCA procedure test accuracy: 0.975 test count: 40


## Your experiment
Before seeing test results, declare a new comparison: change noise or component choices, but keep the split protocol. Use a fresh held-out test set for that revised procedure. Explain why centering on the validation mean would make this a different and leaky pipeline. Submit a shape-annotated forward calculation, an error denominator, and one failure or declared stressful example.
